<a href="https://colab.research.google.com/github/shamilkaperera/Statistical-Learning-e23265/blob/main/Introduction%20to%20Numerical%20Data%20Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundations of Statistical Inference & Hypothesis Testing

## Part A: Maximum Likelihood & Decision Space

**1. MLE Bias and Bessel's Correction**
The MLE for the covariance matrix is $\widehat{\boldsymbol{\Sigma}}_{\text{MLE}} = \frac{1}{n} \sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T$.
Taking the expectation and expanding the terms:
$$ \mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \frac{1}{n} \mathbb{E} \left[ \sum_{i=1}^n (\mathbf{X}_i - \boldsymbol{\mu} - (\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}))(\mathbf{X}_i - \boldsymbol{\mu} - (\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}))^T \right] $$
Expanding this yields:
$$ \mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \frac{1}{n} \left( \sum_{i=1}^n \mathbb{E}[(\mathbf{X}_i - \boldsymbol{\mu})(\mathbf{X}_i - \boldsymbol{\mu})^T] - n \mathbb{E}[(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu})(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu})^T] \right) $$
Since $\mathbb{E}[(\mathbf{X}_i - \boldsymbol{\mu})(\mathbf{X}_i - \boldsymbol{\mu})^T] = \boldsymbol{\Sigma}$ and the variance of the sample mean is $\frac{1}{n}\boldsymbol{\Sigma}$:
$$ \mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \frac{1}{n} \left( n\boldsymbol{\Sigma} - n\left(\frac{1}{n}\boldsymbol{\Sigma}\right) \right) = \frac{n-1}{n}\boldsymbol{\Sigma} $$
Applying Bessel's correction by multiplying by $\frac{n}{n-1}$ gives the unbiased estimator $\mathbf{S}$:
$$ \mathbb{E}[\mathbf{S}] = \frac{n}{n-1} \left( \frac{n-1}{n}\boldsymbol{\Sigma} \right) = \boldsymbol{\Sigma} $$

**2. Type I/II Errors and Conservative Alpha**
*   **Type I Error ($\alpha$):** False alarm (concluding the structure is damaged when it is actually healthy).
*   **Type II Error ($\beta$):** Missed detection (concluding the structure is healthy when it is actually damaged).
*   **Consequence of ultra-conservative $\alpha$ (e.g., 0.0001):** Mathematically, dropping $\alpha$ drastically reduces the statistical power ($1-\beta$) of the test, meaning the probability of missing actual damage (Type II error) increases. Geometrically, an ultra-conservative $\alpha$ massively expands the volume of the "healthy operation" confidence ellipsoid, allowing extreme, anomalous data points to fall inside the boundary and be incorrectly classified as normal.

## Part B: Slutsky's Theorem

**Slutsky’s Theorem:** If a sequence of random vectors $X_n \xrightarrow{d} X$ and a sequence of random matrices $Y_n \xrightarrow{p} c$ (where $c$ is a constant matrix), then $Y_n X_n \xrightarrow{d} cX$.

**Proof of Substitution:**
By the Central Limit Theorem, $\sqrt{n}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \boldsymbol{\Sigma})$.
By the Law of Large Numbers, the sample covariance $\mathbf{S} \xrightarrow{p} \boldsymbol{\Sigma}$ as $n \to \infty$. Consequently, $\mathbf{S}^{-1/2} \xrightarrow{p} \boldsymbol{\Sigma}^{-1/2}$.
Applying Slutsky's Theorem:
$$ \mathbf{S}^{-1/2} \sqrt{n}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \boldsymbol{\Sigma}^{-1/2} \mathscr{N}(\mathbf{0}, \boldsymbol{\Sigma}) = \mathscr{N}(\mathbf{0}, \mathbf{I}) $$
This implies that asymptotically, we can substitute the sample covariance $\mathbf{S}$ for the unknown population covariance $\boldsymbol{\Sigma}$:
$$ \widehat{\boldsymbol{\mu}}_n \sim \mathscr{N}\left(\boldsymbol{\mu}, \frac{1}{n}\mathbf{S}\right) $$

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

np.random.seed(42)
n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01], [0.02, 0.06, 0.03], [0.01, 0.03, 0.05]],
    size=n_samples
)
base_data[4000:, 0] += 0.015
base_data[4000:, 2] -= 0.010
df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])

def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    n, m = df.shape
    chunks = np.array_split(df, g_chunks)
    global_mean = df.mean().values

    W = np.zeros((m, m))
    for chunk in chunks:
        diff = chunk.values - chunk.mean().values
        W += diff.T @ diff

    B = np.zeros((m, m))
    for chunk in chunks:
        diff = chunk.mean().values - global_mean
        B += len(chunk) * np.outer(diff, diff)

    Lambda = np.linalg.det(W) / np.linalg.det(W + B)
    chi2_calc = -(n - 1 - (m + g_chunks) / 2) * np.log(Lambda)
    p_value = 1 - stats.chi2.cdf(chi2_calc, m * (g_chunks - 1))

    conclusion = "Baseline shifted (Not homogeneous)." if p_value < 0.05 else "First moment is homogeneous."

    return {
        "Wilks_Lambda": round(Lambda, 6),
        "Bartlett_Chi2": round(chi2_calc, 4),
        "p_value": round(p_value, 6),
        "Conclusion": conclusion
    }

results = verify_first_moment_homogeneity(df_strain, g_chunks=5)
for k, v in results.items():
    print(f"{k}: {v}")

Wilks_Lambda: 0.990452
Bartlett_Chi2: 47.9215
p_value: 3e-06
Conclusion: Baseline shifted (Not homogeneous).


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


# Geometric Subspace Optimization via PCA

## Part A: Coordinate Projections & Orthogonality

**1. PCA Covariance and Orthogonality**
Given $\mathbf{Z}_i = \mathbf{P}^T \widetilde{\mathbf{X}}_i$, the covariance of the transformed vector is:
$$ \mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] = \mathbb{E}[\mathbf{P}^T \widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T \mathbf{P}] = \mathbf{P}^T \mathbb{E}[\widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T] \mathbf{P} = \mathbf{P}^T \boldsymbol{\Sigma} \mathbf{P} $$
Substituting the spectral decomposition $\boldsymbol{\Sigma} = \mathbf{P} \mathbf{\Lambda} \mathbf{P}^T$:
$$ \mathbf{P}^T (\mathbf{P} \mathbf{\Lambda} \mathbf{P}^T) \mathbf{P} = (\mathbf{P}^T \mathbf{P}) \mathbf{\Lambda} (\mathbf{P}^T \mathbf{P}) = \mathbf{I} \mathbf{\Lambda} \mathbf{I} = \mathbf{\Lambda} $$
Because $\mathbf{\Lambda}$ is strictly diagonal, the off-diagonal elements (cross-covariances) are exactly zero. This means the principal components are completely statistically uncorrelated (orthogonal).

**2. Trace Invariance and Variance Ratios**
Using the cyclic property of the trace ($\text{tr}(\mathbf{A}\mathbf{B}) = \text{tr}(\mathbf{B}\mathbf{A})$):
$$ \text{tr}(\boldsymbol{\Sigma}) = \text{tr}(\mathbf{P} \mathbf{\Lambda} \mathbf{P}^T) = \text{tr}(\mathbf{P}^T \mathbf{P} \mathbf{\Lambda}) = \text{tr}(\mathbf{I} \mathbf{\Lambda}) = \text{tr}(\mathbf{\Lambda}) = \sum_{j=1}^m \lambda_j $$
The mathematical formulations for the variance ratios are:
*   Cumulative Explained Variance Ratio: $\Phi(k) = \frac{\sum_{j=1}^k \lambda_j}{\sum_{j=1}^m \lambda_j}$
*   Residual Unexplained Variance Ratio: $\Psi(k) = \frac{\sum_{j=k+1}^m \lambda_j}{\sum_{j=1}^m \lambda_j} = 1 - \Phi(k)$

**3. Pythagorean Identity and Diagnostics**
The reconstructed vector is $\widehat{\mathbf{x}}_i = \widehat{\boldsymbol{\mu}}_n + \widehat{\mathbf{P}}_k \mathbf{z}_{i,k}$ and the residual is $\mathbf{e}_i = \widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}$.
Because the column spaces of $\widehat{\mathbf{P}}_k$ and $\widehat{\mathbf{P}}_{m-k}$ are orthogonal ($\widehat{\mathbf{P}}_k^T \widehat{\mathbf{P}}_{m-k} = \mathbf{0}$), the cross-term in the norm expansion vanishes:
$$ \|\mathbf{x}_i - \widehat{\boldsymbol{\mu}}_n\|^2 = \|\widehat{\mathbf{P}}_k \mathbf{z}_{i,k}\|^2 + \|\widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}\|^2 = \|\mathbf{z}_{i,k}\|^2 + \|\mathbf{z}_{i,m-k}\|^2 $$
**Diagnostic Contrast:**
*   **Hotelling's $T^2$:** Tracks variation within the principal subspace. It is most sensitive to environmental load anomalies (e.g., extreme wind) which change the correlation structure but keep the data within the expected physical subspace.
*   **$Q$ Statistic (SPE):** Tracks variation in the residual subspace. It is most sensitive to internal structural fractures (e.g., fatigue cracks) which introduce new, unmodeled variance directions that fall outside the trained principal subspace.

In [2]:
def pca_optimization_pipeline(df: pd.DataFrame, k_values=[1, 2, 3]):
    X = df.values
    X_centered = X - np.mean(X, axis=0)
    X_std = X_centered / np.std(X_centered, axis=0, ddof=1)

    S = np.cov(X_std, rowvar=False)
    eigenvalues, P = np.linalg.eigh(S)

    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    P = P[:, idx]

    Z = X_std @ P

    results = {}
    for k in k_values:
        T2 = np.sum(Z[:, :k]**2 / eigenvalues[:k], axis=1)
        Q = np.sum(Z[:, k:]**2, axis=1)

        results[k] = {
            "Mean_T2": round(np.mean(T2), 4),
            "Mean_Q": round(np.mean(Q), 4)
        }
    return results, eigenvalues, P, Z, X_std

np.random.seed(42)
dummy_pca_data = pd.DataFrame(np.random.randn(3000, 4), columns=['S1', 'S2', 'S3', 'S4'])
pca_results, eig_vals, P_mat, Z_mat, X_std_mat = pca_optimization_pipeline(dummy_pca_data, k_values=[1, 2, 3])

print("--- PCA Optimization Pipeline Results ---")
for k, metrics in pca_results.items():
    print(f"Subspace k={k} -> Mean T^2: {metrics['Mean_T2']}, Mean Q: {metrics['Mean_Q']}")

--- PCA Optimization Pipeline Results ---
Subspace k=1 -> Mean T^2: 0.9997, Mean Q: 2.9744
Subspace k=2 -> Mean T^2: 1.9993, Mean Q: 1.9617
Subspace k=3 -> Mean T^2: 2.999, Mean Q: 0.9679


# Latent Subspace Decomposition via Factor Analysis (FA)

## Part A: Generative Model & Commonalities

**1. Fundamental Equation**
Given $\mathbf{Z}_i = \boldsymbol{\Lambda} \mathbf{F}_i + \boldsymbol{\epsilon}_i$. The correlation matrix is:
$$ \mathbf{R} = \mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] = \mathbb{E}[(\boldsymbol{\Lambda} \mathbf{F}_i + \boldsymbol{\epsilon}_i)(\mathbf{F}_i^T \boldsymbol{\Lambda}^T + \boldsymbol{\epsilon}_i^T)] $$
Expanding and using the independence constraint $\mathbb{E}[\boldsymbol{\epsilon}_i \mathbf{F}_i^T] = \mathbf{0}$:
$$ \mathbf{R} = \boldsymbol{\Lambda} \mathbb{E}[\mathbf{F}_i \mathbf{F}_i^T] \boldsymbol{\Lambda}^T + \mathbb{E}[\boldsymbol{\epsilon}_i \boldsymbol{\epsilon}_i^T] = \boldsymbol{\Lambda} \mathbf{I} \boldsymbol{\Lambda}^T + \boldsymbol{\Psi} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi} $$
*   **Communality ($h_j^2$):** The $j$-th diagonal element of $\boldsymbol{\Lambda}\boldsymbol{\Lambda}^T$. It represents the variance of sensor $j$ explained by the common latent physical factors.
*   **Uniqueness ($\varphi_j^2$):** The $j$-th diagonal element of $\boldsymbol{\Psi}$. It represents the variance of sensor $j$ that is unique to that sensor (localized noise/error) and unexplained by the common factors.

**2. Varimax Rotation Invariance**
Varimax rotation maximizes the variance of the squared loadings to create a "simple structure" (clean, isolated groupings), unlike raw PCA loadings which are mathematically ordered by eigenvalue magnitude.
Let $\mathbf{T}$ be an orthogonal rotation matrix ($\mathbf{T}\mathbf{T}^T = \mathbf{I}$). The rotated loadings are $\boldsymbol{\Lambda}^* = \boldsymbol{\Lambda}\mathbf{T}$.
The new communality matrix is:
$$ \boldsymbol{\Lambda}^* (\boldsymbol{\Lambda}^*)^T = (\boldsymbol{\Lambda}\mathbf{T})(\boldsymbol{\Lambda}\mathbf{T})^T = \boldsymbol{\Lambda}\mathbf{T}\mathbf{T}^T\boldsymbol{\Lambda}^T = \boldsymbol{\Lambda}\mathbf{I}\boldsymbol{\Lambda}^T = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T $$
Because $\mathbf{T}$ is orthogonal, the row sums of squares (communalities) and the global covariance approximation remain completely unchanged.

**3. Thomson’s Regression Method**
Construct the joint vector $\mathbf{Y}_i = [\mathbf{Z}_i^T, \mathbf{F}_i^T]^T$. Its covariance is:
$$ \boldsymbol{\Sigma}_{YY} = \begin{bmatrix} \mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] & \mathbb{E}[\mathbf{Z}_i \mathbf{F}_i^T] \\ \mathbb{E}[\mathbf{F}_i \mathbf{Z}_i^T] & \mathbb{E}[\mathbf{F}_i \mathbf{F}_i^T] \end{bmatrix} $$
Since $\mathbb{E}[\mathbf{Z}_i \mathbf{F}_i^T] = \mathbb{E}[(\boldsymbol{\Lambda} \mathbf{F}_i + \boldsymbol{\epsilon}_i) \mathbf{F}_i^T] = \boldsymbol{\Lambda} \mathbf{I} = \boldsymbol{\Lambda}$, we get:
$$ \boldsymbol{\Sigma}_{YY} = \begin{bmatrix} \mathbf{R} & \boldsymbol{\Lambda} \\ \boldsymbol{\Lambda}^T & \mathbf{I} \end{bmatrix} $$
Using the conditional mean formula for multivariate normals $\mathbb{E}[\mathbf{X}_2 | \mathbf{x}_1] = \boldsymbol{\mu}_2 + \boldsymbol{\Sigma}_{21} \boldsymbol{\Sigma}_{11}^{-1} (\mathbf{x}_1 - \boldsymbol{\mu}_1)$ with zero means:
$$ \mathbf{f}_i = \mathbb{E}[\mathbf{F}_i | \mathbf{Z}_i] = \mathbf{0} + \boldsymbol{\Lambda}^T \mathbf{R}^{-1} (\mathbf{z}_i - \mathbf{0}) = \boldsymbol{\Lambda}^T \mathbf{R}^{-1} \mathbf{z}_i $$

In [3]:
!pip install factor_analyzer -q
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer
from factor_analyzer.rotator import Rotator
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def fa_engine_and_dashboard(df, k):
    n, m = df.shape
    if n <= m: raise ValueError("Snapshot count (n) must be strictly greater than channel count (m).")
    if k >= m: raise ValueError("Latent space dimension (k) must be strictly less than feature space dimension (m).")

    scaler = StandardScaler()
    Z = scaler.fit_transform(df)
    std_devs = np.std(df, axis=0, ddof=1)
    if np.any(std_devs < 1e-15): Z[:, std_devs < 1e-15] = 0

    fa = FactorAnalyzer(n_factors=k, method='ml', rotation=None)
    fa.fit(Z)
    rotator = Rotator(method='varimax')
    loadings_rotated = rotator.fit_transform(fa.loadings_)

    h2 = np.sum(loadings_rotated**2, axis=1)
    phi2 = 1 - h2

    R = np.corrcoef(Z, rowvar=False)
    F = Z @ np.linalg.inv(R) @ loadings_rotated

    print(f"--- FA Telemetry ---")
    print(f"Average System Communality %: {np.mean(h2) * 100:.2f}%")
    print(f"Average System Uniqueness %: {np.mean(phi2) * 100:.2f}%\n")

    fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.24, vertical_spacing=0.28,
                        subplot_titles=("Structural Loadings Heatmap", "Variance Partitioning Profile",
                                        "Sensor Uniqueness Line Profile", "Latent Factor Variance Distribution"))
    fig.update_layout(width=1250, height=750, template="plotly_white",
                      legend=dict(orientation="h", x=0.5, y=1.02, xanchor="center"),
                      margin=dict(t=150, b=60, l=140, r=80))
    sensors = df.columns

    fig.add_trace(go.Heatmap(z=np.abs(loadings_rotated), x=[f"Factor {i+1}" for i in range(k)], y=sensors,
                             colorscale='YlOrRd', showscale=True,
                             colorbar=dict(x=-0.15, len=0.38, y=0.78, yanchor="middle", xanchor="right", title="|Loadings|")), row=1, col=1)

    fig.add_trace(go.Bar(y=sensors, x=h2*100, name='Communality (h²)', marker_color='#1f77b4', orientation='h'), row=1, col=2)
    fig.add_trace(go.Bar(y=sensors, x=phi2*100, name='Uniqueness (φ²)', marker_color='#ff7f0e', orientation='h'), row=1, col=2)
    fig.update_xaxes(range=[0, 100], title_text="% Variance", row=1, col=2)

    fig.add_trace(go.Scatter(x=sensors, y=phi2, mode='lines+markers', name='Uniqueness',
                             line=dict(color='#d62728', width=2, dash='dashdot'), marker=dict(symbol='x', size=8)), row=2, col=1)
    fig.update_xaxes(tickangle=25, title_text="Sensor Channel", row=2, col=1)
    fig.update_yaxes(title_text="φ²", row=2, col=1)

    variances = np.var(F, axis=0, ddof=1)
    fig.add_trace(go.Bar(x=[f"Factor {i+1}" for i in range(k)], y=variances, name='Factor Variance',
                         marker_color='#2ca02c', marker_line=dict(width=0.5, color='black')), row=2, col=2)
    fig.update_yaxes(title_text="Empirical Variance", row=2, col=2)

    fig.show()
    return loadings_rotated, h2, phi2, F

np.random.seed(42)
dummy_fa_data = pd.DataFrame(np.random.randn(500, 4), columns=['S1', 'S2', 'S3', 'S4'])
fa_engine_and_dashboard(dummy_fa_data, k=2)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
--- FA Telemetry ---
Average System Communality %: 49.80%
Average System Uniqueness %: 50.20%



(array([[-0.9974382 ,  0.01081842],
        [ 0.02427863, -0.02179028],
        [ 0.06562301,  0.99533593],
        [-0.01722516,  0.02875091]]),
 array([0.995     , 0.00106427, 0.995     , 0.00112332]),
 array([0.005     , 0.99893573, 0.005     , 0.99887668]),
 array([[-4.98762207e-01,  6.44233990e-01],
        [ 3.00235250e-01,  1.52573983e+00],
        [ 5.31568246e-01, -5.37883521e-01],
        [-2.50386811e-01, -1.75122720e+00],
        [ 1.11291322e+00, -1.02197109e+00],
        [-1.55038138e+00,  1.31129105e-01],
        [ 6.04591181e-01, -1.23195653e+00],
        [ 6.71224449e-01, -6.85300632e-01],
        [ 5.38062887e-02,  7.83002186e-01],
        [-2.10328370e-01, -1.35593551e+00],
        [-7.67899133e-01, -1.03798189e-01],
        [ 1.61999041e+00, -6.06200786e-01],
        [-3.37219433e-01,  3.09050545e-01],
        [ 7.71930716e-01,  9.45003520e-01],
        [ 9.39114473e-01,  2.32495896e-01],
        [ 5.34841316e-01, -1.18278719e+00],
        [-8.47288175e-01, -5.47728

In [5]:
def pca_optimization_dashboard(df):
    m = df.shape[1]
    sensors = df.columns

    _, eigenvalues, P, Z, _ = pca_optimization_pipeline(df, k_values=list(range(1, m)))

    fig = make_subplots(rows=2, cols=3,
                        subplot_titles=("Feature Loadings Matrix", "Absolute Eigenvalues", "Explained Variance",
                                        "Residual Space", "Mean Hotelling's T^2", "Mean Q Statistic"))
    fig.update_layout(width=1400, height=800, template="plotly_white", showlegend=False, margin=dict(t=100, b=50, l=50, r=50))

    fig.add_trace(go.Heatmap(z=np.abs(P), x=[f"PC {i+1}" for i in range(m)], y=sensors, colorscale='Viridis'), row=1, col=1)

    fig.add_trace(go.Bar(x=[f"PC {i+1}" for i in range(m)], y=eigenvalues, marker_color='blue'), row=1, col=2)

    marginal = (eigenvalues / np.sum(eigenvalues)) * 100
    cumulative = np.cumsum(marginal)
    fig.add_trace(go.Bar(x=[f"PC {i+1}" for i in range(m)], y=marginal, name='Marginal %'), row=1, col=3)
    fig.add_trace(go.Scatter(x=[f"PC {i+1}" for i in range(m)], y=cumulative, mode='lines+markers', name='Cum %', line=dict(dash='dash')), row=1, col=3)

    residual = 100 - cumulative
    fig.add_trace(go.Bar(x=[f"PC {i+1}" for i in range(m)], y=residual, marker_color='red'), row=2, col=1)

    T2_means, Q_means = [], []
    for k in range(1, m):
        T2 = np.sum(Z[:, :k]**2 / eigenvalues[:k], axis=1)
        Q = np.sum(Z[:, k:]**2, axis=1)
        T2_means.append(np.mean(T2))
        Q_means.append(np.mean(Q))

    fig.add_trace(go.Scatter(x=[f"k={i+1}" for i in range(m-1)], y=T2_means, mode='lines+markers', line=dict(color='green')), row=2, col=2)
    fig.add_trace(go.Scatter(x=[f"k={i+1}" for i in range(m-1)], y=Q_means, mode='lines+markers', line=dict(color='orange')), row=2, col=3)

    fig.show()

In [6]:
np.random.seed(42)
n_samples = 2500

f1 = np.random.normal(0, 1, n_samples)
f2 = np.random.normal(0, 1, n_samples)

s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)

df_asset = pd.DataFrame(
    data=np.vstack([s1, s2, s3, s4]).T,
    columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4']
)

print("--- Generating PCA 2x3 Dashboard ---")
pca_optimization_dashboard(df_asset)

print("\n--- Generating FA 2x2 Dashboard ---")
fa_engine_and_dashboard(df_asset, k=2)

--- Generating PCA 2x3 Dashboard ---



--- Generating FA 2x2 Dashboard ---
--- FA Telemetry ---
Average System Communality %: 58.60%
Average System Uniqueness %: 41.40%



(array([[0.98659669, 0.14706109],
        [0.83253959, 0.25797797],
        [0.08503271, 0.75909764],
        [0.02527073, 0.0717343 ]]),
 array([0.99499999, 0.75967481, 0.58345978, 0.00578442]),
 array([0.00500001, 0.24032519, 0.41654022, 0.99421558]),
 array([[ 0.31199012,  0.544632  ],
        [-0.29030904,  0.64329538],
        [-0.01419905, -0.10451341],
        ...,
        [ 0.44296321,  2.19656802],
        [-0.21818619,  0.84765889],
        [ 0.26226797, -0.71822048]]))

# Subspace Diagnostics & Feature De-correlation

## Part B: Analysis

**Q1: The Total Variance Illusion**
1. A Uniqueness value ($\varphi^2$) near 100% for `Sensor_4` means almost all of its variance is localized, random noise. It is completely disconnected from the true physical processes ($f_1$ and $f_2$) driving the rest of the asset.
2. PCA blindly maximizes total global variance. Because `Sensor_4` has massive localized noise ($\sigma^2 \approx 2.0$), PCA incorrectly treats this huge noise variance as a "principal component" and inflates its top eigenvalues to capture it.
3. **Risk:** If an engineer relies only on PCA, the anomaly detection framework will be heavily skewed by sensor noise. The model will trigger false alarms during normal noise fluctuations, or fail to detect actual structural damage because the principal components are busy tracking electrical noise rather than physical health.

**Q2: Decoupling Structural Loading via Rotation**
1. Traditional PCA forces a strict mathematical hierarchy where every component sweeps up a mix of all variances. Varimax rotation relaxes this by maximizing the variance of the squared loadings. This mathematically forces the loadings to be either very high or very low (approaching 0 or 1), creating a "simple structure" where sensors cleanly group into isolated physical factors.
2. From an operator's standpoint, the rotated FA heatmap is vastly easier to troubleshoot. It clearly isolates the physical modes (e.g., Factor 1 = primary structural mode, Factor 2 = secondary operational mode). If an anomaly occurs, the operator can instantly see which physical mode is affected. Raw PCA eigenvectors are mathematically mixed, making root-cause analysis nearly impossible.

**Q3: Determining Subspace Truncation ($k$)**
1. As the cutoff transitions from $k=1$ to $k=2$, the Mean $Q$ Statistic drops sharply because the second true physical dimension is captured. From $k=2$ to $k=3$, the curve flattens out into a distinct "elbow" because the remaining dimensions consist purely of random noise.
2. The sharp drop followed by a flat elbow at $k=2$ identifies the true hidden physical dimensionality. It shows that exactly 2 latent factors govern the structural physics, and anything beyond that is just residual noise.
3. If you incorrectly choose $k=3$, you are forcing random sensor noise (specifically from the faulty `Sensor_4`) into your "clean" principal subspace. This corrupts the model, making your monitoring system highly sensitive to normal electrical noise.

**Q4: Operational Trade-offs in System Health Monitoring**
The **FA Strategy** is significantly more robust against a single sensor losing calibration or experiencing an electrical short.
*   **Justification:** Factor Analysis explicitly models the Sensor Uniqueness Noise Floor ($\varphi^2$). If `Sensor_4` shorts out, its uniqueness metric will spike to near 100%, but the common latent factors (which represent the true structural health of the asset) will remain stable and unaffected.
*   Conversely, the PCA Strategy mixes the faulty sensor's massive variance into all principal components. This immediately corrupts the $T^2$ and $Q$ statistics for the entire system, triggering widespread false alarms and rendering the fleet monitoring pipeline useless.